# Image Segmentation with CLIPSeg

This notebook performs image segmentation using the `CIDAS/clipseg-rd64-refined` model from Hugging Face.


In [ ]:
from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation
from PIL import Image
import torch
import os
import matplotlib.pyplot as plt

# Load model and processor
processor = CLIPSegProcessor.from_pretrained('CIDAS/clipseg-rd64-refined')
model = CLIPSegForImageSegmentation.from_pretrained('CIDAS/clipseg-rd64-refined')

# Directory containing images
image_dir = 'img'
image_files = [os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.endswith(('jpg', 'png'))]

# --- Prompts for segmentation ---
prompts_per_image = {
    'bird.jpg': ['a bird'],
    'cats_dog.png': ['a cat', 'a dog'],
    'kitchen.png': ['a vase', 'a bottle', 'potted plant', 'oven', 'bowl'],
    'pizza.png': ['pizza']
}

for image_path in image_files:
    image_name = os.path.basename(image_path)
    prompts = prompts_per_image.get(image_name, [])

    if not prompts:
        print(f'No prompts defined for {image_name}, skipping.')
        continue

    print(f'Processing image: {image_path} with prompts: {prompts}')
    image = Image.open(image_path).convert('RGB')

    # Predict segmentation maps
    inputs = processor(text=prompts, images=[image] * len(prompts), padding='max_length', return_tensors='pt')
    with torch.no_grad():
        outputs = model(**inputs)
    preds = outputs.logits

    # Visualize the results
    fig, ax = plt.subplots(1, len(prompts) + 1, figsize=(3 * (len(prompts) + 1), 4))
    
    # Display original image
    ax[0].imshow(image)
    ax[0].axis('off')
    ax[0].set_title('Original Image')

    # Display segmentation masks
    for i in range(len(prompts)):
        ax[i+1].imshow(torch.sigmoid(preds[i]))
        ax[i+1].axis('off')
        ax[i+1].set_title(prompts[i])

    plt.tight_layout()
    plt.show()
    print('-' * 30)


: 